# NB3 — FastAPI `/search` Endpoint + Latency Benchmark

**Stack:** FastAPI + uvicorn + httpx (client). Searcher từ `app/search.py`.
Maps to slide §7 (Production Patterns) + deliverable bullets 1, 4.

> Mục tiêu: bọc `Searcher` thành REST API, đo P50/P95/P99 latency, đảm bảo
> P99 < 50 ms cho hybrid mode (rubric threshold).

In [1]:
import _setup  # noqa: F401
import statistics
import time
from pathlib import Path

import httpx
from fastapi.testclient import TestClient

from app.main import app
import app.main as main_module

F:\VinUni_Lab\VinUni_Day19\2A202600651-LeTriNguyen-Day19-Track2-VectorFeatureStore-Lab\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


F:\VinUni_Lab\VinUni_Day19\2A202600651-LeTriNguyen-Day19-Track2-VectorFeatureStore-Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Khởi động API client

Trong production thực tế, bạn sẽ chạy `make api` ở terminal riêng. Để notebook
này chạy ổn định trên Windows và trong sandbox, ta dùng `TestClient` để gọi
trực tiếp FastAPI app với cùng response model và lifecycle startup/shutdown.

In [2]:
ROOT = Path(_setup.__file__).resolve().parent.parent
client = TestClient(app)
client.__enter__()
health = client.get("/healthz")
health.raise_for_status()
print(health.json())

for _ in range(10):
    warm = client.get("/search", params={"q": "cloud computing tu dong mo rong", "mode": "hybrid"})
    warm.raise_for_status()
for _ in range(5):
    warm = client.get("/search", params={"q": "oauth jwt zero-trust", "mode": "semantic"})
    warm.raise_for_status()
for _ in range(5):
    warm = client.get("/search", params={"q": "terraform gitops", "mode": "keyword"})
    warm.raise_for_status()

print("Warm-up complete")

{'ready': True, 'n_docs': 1000}


Warm-up complete


## 2. Single query — kiểm tra response shape

In [3]:
r = client.get("/search", params={"q": "cloud computing tự động mở rộng", "mode": "hybrid"})
r.raise_for_status()
body = r.json()
print(f"latency_ms: {body['latency_ms']:.1f}")
print(f"top-3 hits:")
for h in body["hits"][:3]:
    print(f"  {h['doc_id']:>14}  score={h['score']:.4f}  {h['title']}")

latency_ms: 11.3
top-3 hits:
       cloud_016  score=0.0325  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_072  score=0.0323  Điện toán đám mây: tự động mở rộng theo lưu lượng
       cloud_053  score=0.0315  Điện toán đám mây: tự động mở rộng theo lưu lượng


## 3. TODO — Latency benchmark (100 queries × 3 modes)

Dùng 50 golden queries × 2 reps = 100 calls/mode. Ghi nhận latency từ
`body["latency_ms"]` (server-side, đã trừ network) HOẶC từ wall-clock httpx
(bao gồm network) — note: rubric assert P99 < 50ms áp dụng cho server-side.

Output: bảng P50/P95/P99 cho 3 mode.

In [4]:
import json

DATA = ROOT / "data"
golden = [json.loads(l) for l in (DATA / "golden_set.jsonl").open(encoding="utf-8")]


def percentile(values: list[float], p: float) -> float:
    n = len(values)
    if n == 0:
        return 0.0
    return sorted(values)[min(int(n * p), n - 1)]


def benchmark_mode(mode: str, reps: int = 2) -> dict[str, float]:
    searcher = main_module._searcher
    assert searcher is not None
    server_latencies: list[float] = []
    wall_latencies: list[float] = []
    for q in golden[:10]:
        searcher.search(q["query"], mode=mode)
        warm = client.get("/search", params={"q": q["query"], "mode": mode})
        warm.raise_for_status()
    for _ in range(reps):
        for q in golden:
            t0 = time.perf_counter()
            searcher.search(q["query"], mode=mode)
            server_latencies.append((time.perf_counter() - t0) * 1000)

            t0 = time.perf_counter()
            r = client.get("/search", params={"q": q["query"], "mode": mode})
            wall_latencies.append((time.perf_counter() - t0) * 1000)
    return {
        "p50_server": percentile(server_latencies, 0.50),
        "p95_server": percentile(server_latencies, 0.95),
        "p99_server": percentile(server_latencies, 0.99),
        "p99_wall":   percentile(wall_latencies, 0.99),
    }


print(f"  {'mode':10}  {'P50':>7}  {'P95':>7}  {'P99':>7}  {'P99(wall)':>9}")
results = {}
for mode in ("keyword", "semantic", "hybrid"):
    res = benchmark_mode(mode)
    results[mode] = res
    print(f"  {mode:10}  {res['p50_server']:>5.1f}ms  {res['p95_server']:>5.1f}ms  "
          f"{res['p99_server']:>5.1f}ms  {res['p99_wall']:>7.1f}ms")

  mode            P50      P95      P99  P99(wall)


  keyword       1.9ms    3.9ms    5.0ms     10.1ms


  semantic      9.2ms   13.0ms   14.5ms     17.8ms


  hybrid       12.4ms   16.3ms   20.7ms     24.3ms


## 4. Rubric assertion — hybrid P99 server-side < 50ms

In [5]:
hybrid_p99 = results["hybrid"]["p99_server"]
print(f"Hybrid P99 server-side: {hybrid_p99:.1f}ms")
if hybrid_p99 < 50:
    print(f"PASS — hybrid P99 < 50ms ({hybrid_p99:.1f}ms)")
else:
    print(f"WARN — hybrid P99 >= 50ms ({hybrid_p99:.1f}ms)")
    print("  Possible causes: cold cache, fastembed model not warm yet, or RRF depth=50 is too aggressive")
    print("  Check: re-run benchmark after 10 warm-up queries; or reduce RRF depth")

Hybrid P99 server-side: 20.7ms
PASS — hybrid P99 < 50ms (20.7ms)


## 5. Cleanup — close the API client

In [6]:
client.__exit__(None, None, None)
print("API client closed")

API client closed


## Deliverable evidence

1. Output cell 2: 1 single hybrid query response with `top-3 hits`.
2. Output cell 3: latency table P50/P95/P99 for keyword/semantic/hybrid.
3. Output cell 4: hybrid P99 < 50ms PASS.

---

## Vibe-coding callout

**Delegate freely:** the FastAPI scaffolding (route definition, Pydantic
response model, lifespan handler). AI generates this perfectly given the
spec "GET /search?q=str&mode=Literal[...] returning SearchResponse with
latency_ms field". `app/main.py` is exactly that pattern — review the diff,
don't write it from scratch.

**Think hard yourself:** *what to measure*. Server-side latency vs wall-clock
vs client-side. P50 vs P95 vs P99. Cold vs warm. Single user vs concurrent.
These are *judgement* decisions: nếu rubric chỉ check P99, optimization sẽ
hướng vào tail latency, không phải mean. Đừng nhờ AI quyết định metric —
chỉ nhờ implement metric đã chọn.